# The Geometry of Superposition

*Book two of the superposition series. Book one
([`toy_models_of_superposition.ipynb`](./toy_models_of_superposition.ipynb)) showed that
sparsity turns superposition on. This notebook asks the question book one left open:
when a model superposes features, **what shape do they make — and why?***

Reference: Elhage et al. 2022, [*Toy Models of
Superposition*](https://transformer-circuits.pub/2022/toy_model/index.html) — the
"Geometry of Superposition" and "Learning Dynamics" sections. Hyperparameters are pinned
from the paper's public Colab.

**Hypothesis.** Superposition is not amorphous. Features arrange into *uniform
polytopes* — digons, triangles, tetrahedra, pentagons, square antiprisms — measurable as
fractional *feature dimensionality* clinging to ½, ⅔, ¾, ⅖, ⅜. Training reaches these
configurations through discrete *energy-level jumps*, and non-uniformity deforms the
geometry smoothly until it snaps.

*Scope note: like book one, this notebook is pure synthetic toy — no real language
model, and no Arabic; the dialect thread resumes once these tools meet real models.*

## Act 0 — The question book one left open

Book one ended on a pentagon: five features, two hidden dimensions, high sparsity — and
the trained weight columns landed at five *equal* angles, 72° apart. We treated that as
an observation. But nothing in the loss says "be regular." Why not four features crammed
and one straggler? Why the most symmetric arrangement available?

Here is the tease: our model is solving a physics problem. Place charged particles on a
sphere and let them repel — they settle into maximally-symmetric configurations (the
[Thomson problem](https://en.wikipedia.org/wiki/Thomson_problem)). Interference between
features acts like repulsion between charges. By the end of this notebook that analogy
will be quantitative: we will *measure* the fraction of a dimension each feature gets,
watch those fractions cling to a handful of exact values (½, ⅔, ¾, ⅖, ⅜), and identify
the polytope behind each value.

The plan: rebuild the toy (Act 1) → build the measuring instrument and calibrate it on
known solutions (Act 2) → sweep sparsity and find the plateaus (Act 3) → show the
plateaus are polytopes (Act 4) → watch training jump between them (Act 5) → perturb one
feature and watch the geometry stretch and snap (Act 6).

In [ ]:
# lib: imports
from pathlib import Path

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

SEED = 0
torch.manual_seed(SEED)
CACHE = Path("cache")  # created by the cells that write to it (keeps pytest's cwd clean)
print("torch", torch.__version__, "| seed", SEED)

## Act 1 — Rebuild the toy

Same model as book one, restated so this notebook stands alone: embed `n` features into
`m < n` dimensions through `W [m, n]`, read back through `Wᵀ`, add a bias, ReLU. Data is
sparse — each feature is 0 with probability `S`, else uniform on [0, 1). One deliberate
change from book one: **importance is uniform** (`Iᵢ = 1`). The geometry section of the
paper studies *uniform* superposition — all features identical — because symmetric
problems have symmetric solutions, and that symmetry is exactly what we want to explain.

Shapes to hold onto: `x [B, n] → h = xWᵀ [B, m] → x' = ReLU(hW + b) [B, n]`.

First, the calibration case: n=5, m=2 at density `1−S = 0.05` (the paper's setting for
this small model). If the pentagon reproduces, we have our known-good solution.

In [ ]:
# lib: make_batch
def make_batch(n_features, sparsity, batch_size, generator=None):
    """Sparse synthetic features.

    Each entry is 0 with probability `sparsity`, otherwise uniform on [0, 1).
    Returns a tensor of shape [batch_size, n_features].
    """
    vals = torch.rand(batch_size, n_features, generator=generator)
    keep = torch.rand(batch_size, n_features, generator=generator) >= sparsity
    return vals * keep

In [ ]:
# lib: toymodel
class ToyModel(torch.nn.Module):
    """Embed n features into m<n dims via W [m, n], read back through Wᵀ, add bias.

    forward: h = x @ W.T ; out = h @ W + b ; ReLU(out) if use_relu else out.
    """
    def __init__(self, n_features, n_hidden, use_relu=True):
        super().__init__()
        self.use_relu = use_relu
        self.W = torch.nn.Parameter(torch.empty(n_hidden, n_features))
        torch.nn.init.xavier_normal_(self.W)
        self.b = torch.nn.Parameter(torch.zeros(n_features))

    def forward(self, x):
        h = x @ self.W.T           # [B, m]
        out = h @ self.W + self.b  # [B, n]
        return F.relu(out) if self.use_relu else out

In [ ]:
# lib: train
def train(model, sparsity, importance, steps=10_000, lr=1e-3, batch_size=1024, seed=0):
    """Train `model` to reconstruct sparse features under importance-weighted MSE.

    Loss = mean over batch of Σᵢ importanceᵢ · (xᵢ − x'ᵢ)². Returns loss sampled every
    500 steps.
    """
    n_features = model.W.shape[1]
    gen = torch.Generator().manual_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    losses = []
    for step in range(steps):
        x = make_batch(n_features, sparsity, batch_size, generator=gen)
        out = model(x)
        loss = (importance * (x - out) ** 2).sum(dim=-1).mean()
        opt.zero_grad()
        loss.backward()
        opt.step()
        if step % 500 == 0:
            losses.append(loss.item())
    return losses

In [ ]:
# lib: plot_features_2d
def plot_features_2d(W, ax=None, title=None):
    """Draw each column of a [2, n] weight matrix as a ray from the origin."""
    Wn = W.detach().cpu().numpy()
    ax = ax or plt.gca()
    for i in range(Wn.shape[1]):
        ax.plot([0.0, Wn[0, i]], [0.0, Wn[1, i]], marker="o")
    lim = float(np.abs(Wn).max()) * 1.1 + 1e-9
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.set_aspect("equal")
    ax.axhline(0, lw=0.5, color="gray"); ax.axvline(0, lw=0.5, color="gray")
    if title:
        ax.set_title(title)

In [ ]:
# lib: plot_WtW
def plot_WtW(W, ax=None, title=None):
    """Heatmap of WᵀW: diagonal = how strongly each feature is represented,
    off-diagonal = interference between features."""
    WtW = (W.T @ W).detach().cpu().numpy()
    ax = ax or plt.gca()
    im = ax.imshow(WtW, cmap="RdBu", vmin=-1.0, vmax=1.0)
    ax.set_xticks(range(WtW.shape[0])); ax.set_yticks(range(WtW.shape[0]))
    if title:
        ax.set_title(title)
    return im

In [ ]:
# Act 1: the calibration pentagon — n=5, m=2, uniform importance, density 1−S = 0.05
UNIFORM_DENSITY = 0.05

torch.manual_seed(SEED)
pentagon = ToyModel(5, 2)
losses = train(pentagon, sparsity=1 - UNIFORM_DENSITY, importance=torch.ones(5), seed=SEED)

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
plot_features_2d(pentagon.W, ax=axes[0], title="W columns — the pentagon returns")
plot_WtW(pentagon.W, ax=axes[1], title="WᵀW")
plt.tight_layout(); plt.show()

W_p = pentagon.W.detach()
angles = torch.atan2(W_p[1], W_p[0]).rad2deg().sort().values
gaps = torch.diff(torch.cat([angles, angles[:1] + 360]))
print("column norms:", [f"{v:.3f}" for v in W_p.norm(dim=0)])
print("angular gaps (deg):", [f"{v:.1f}" for v in gaps], "| regular pentagon = 72.0 each")

Whatever the run produced is the result: if the gaps sit near 72° with roughly equal
norms, we have the regular pentagon and a calibration target. If the run landed in a
different configuration (these toys have local minima — Act 5 is about exactly that),
we note what formed and continue; the instrument in Act 2 works either way.

## Act 2 — The instrument: feature dimensionality

The paper's measuring device. Define the **dimensionality of feature i** as

$$D_i = \frac{\lVert W_i \rVert^2}{\sum_j (\hat W_i \cdot W_j)^2}$$

Numerator: how strongly feature *i* is represented. Denominator: how many features share
the direction it lives in (each `Wⱼ` projected onto `Ŵᵢ`, squared, summed — the `j = i`
term contributes `‖Wᵢ‖²`, so a feature always shares with at least itself). Read it as
"the fraction of a dimension feature *i* gets to keep."

Predictions before we measure — the name-then-experiment ritual:

| configuration | prediction | arithmetic |
|---|---|---|
| dedicated orthogonal direction | 1 | ‖Wᵢ‖²/‖Wᵢ‖² |
| antipodal pair (digon) | ½ | 1/(1+1) |
| regular pentagon vertex | ⅖ | 1/(1 + 2cos²72° + 2cos²144°) = 1/2.5 |
| dropped feature | 0 | 0/anything |

A companion summary statistic: `D* = m/‖W‖²_F`, "dimensions per feature." Since
represented features have `‖Wᵢ‖ ≈ 1` and dropped ones `≈ 0`, `‖W‖²_F` counts learned
features, and `D*` is the budget each one gets on average.

The instrument earns trust only by reading the known cases correctly — hand-built exact
configurations first, then the pentagon we actually trained in Act 1.

In [ ]:
# lib: feature_dimensionality
def feature_dimensionality(W, eps=1e-6):
    """Dᵢ = ‖Wᵢ‖² / Σⱼ (Ŵᵢ·Wⱼ)², W [m, n] (paper's compute_dimensionality).

    Features with ~zero norm are defined as D = 0 (unlearned) rather than 0/0.
    """
    W = W.detach()
    norms = W.norm(dim=0)                            # [n]
    W_unit = W / norms.clamp(min=eps)                # [m, n]
    interference = ((W_unit.T @ W) ** 2).sum(dim=1)  # [n]
    D = norms ** 2 / interference.clamp(min=eps)
    return torch.where(norms > eps, D, torch.zeros_like(D))

In [ ]:
# lib: frobenius_dims_per_feature
def frobenius_dims_per_feature(W):
    """D* = m / ‖W‖²_F — average dimensions per learned feature."""
    W = W.detach()
    return W.shape[0] / (W.norm() ** 2).item()

In [ ]:
# Act 2: calibrate the instrument on known cases
exact_pentagon = torch.stack([
    torch.cos(2 * torch.pi * torch.arange(5) / 5),
    torch.sin(2 * torch.pi * torch.arange(5) / 5),
])
cases = {
    "identity (4 dedicated dims)": torch.eye(4),
    "antipodal pair": torch.tensor([[1.0, -1.0]]),
    "exact pentagon": exact_pentagon,
    "trained pentagon (Act 1)": pentagon.W,
}
for name, W in cases.items():
    D = feature_dimensionality(W)
    print(f"{name:32s} D = {[f'{d:.3f}' for d in D]}  D* = {frobenius_dims_per_feature(W):.3f}")

The exact cases must land on 1, ½, ⅖ by arithmetic; the *trained* pentagon is the real
test — its Dᵢ should sit near 0.4 without ever being told about pentagons. The
instrument now scales to any `W`, which is what Act 3 needs: 200 features, 20 dims, 20
sparsities at once.

### Interlude — twenty models for the price of one

Act 3 needs one trained model per sparsity value. Trained one at a time that is 20 runs;
the paper's Colab instead gives `W` a leading *instance* dimension — `W [I, m, n]` — and
trains all twenty models in a single loop, each instance seeing its own sparsity. Same
model, vectorized; einsum keeps each instance's forward pass separate. (Training
follows the Colab exactly: AdamW, constant lr 1e-3, uniform importance, 10k steps,
batch 1024 — and the loss is *averaged* over features per instance, then summed over
instances, so instances don't trade off against each other.)

In [ ]:
# lib: batched_toymodel
class BatchedToyModel(torch.nn.Module):
    """I independent ToyModels trained at once: W [I, m, n], b [I, n].

    forward: x [B, I, n] → h [B, I, m] → ReLU(out) [B, I, n]. Instance i never mixes
    with instance j — the einsums contract only within an instance.
    """
    def __init__(self, n_instances, n_features, n_hidden):
        super().__init__()
        self.W = torch.nn.Parameter(torch.empty(n_instances, n_hidden, n_features))
        for i in range(n_instances):
            torch.nn.init.xavier_normal_(self.W.data[i])
        self.b = torch.nn.Parameter(torch.zeros(n_instances, n_features))

    def forward(self, x):
        h = torch.einsum("bif,imf->bim", x, self.W)
        out = torch.einsum("bim,imf->bif", h, self.W) + self.b
        return F.relu(out)

In [ ]:
# lib: make_batch_batched
def make_batch_batched(n_instances, n_features, sparsities, batch_size, generator=None):
    """Per-instance sparse features: sparsities [I] of S values → batch [B, I, n]."""
    vals = torch.rand(batch_size, n_instances, n_features, generator=generator)
    keep = (
        torch.rand(batch_size, n_instances, n_features, generator=generator)
        >= sparsities[None, :, None]
    )
    return vals * keep

In [ ]:
# lib: train_batched
def train_batched(model, sparsities, steps=10_000, lr=1e-3, batch_size=1024, seed=0,
                  snapshot_every=None):
    """Colab-faithful training: AdamW, constant lr, importance ≡ 1.

    Loss = per-instance mean over batch and features, summed over instances.
    Optionally clones W every `snapshot_every` steps (plus the last step).
    Returns dict(losses=[(step, loss)], snap_steps=[...], snapshots=[W clones]).
    """
    n_instances, _, n_features = model.W.shape
    gen = torch.Generator().manual_seed(seed)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    log = {"losses": [], "snap_steps": [], "snapshots": []}
    for step in range(steps):
        x = make_batch_batched(n_instances, n_features, sparsities, batch_size, generator=gen)
        out = model(x)
        loss = ((x - out) ** 2).mean(dim=(0, 2)).sum()
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        if step % 100 == 0 or step == steps - 1:
            log["losses"].append((step, loss.item()))
        if snapshot_every and (step % snapshot_every == 0 or step == steps - 1):
            log["snap_steps"].append(step)
            log["snapshots"].append(model.W.detach().clone())
    return log

## Act 3 — The uniform sweep

The paper's setup, from its Colab: **n = 200 features, m = 20 dims, 20 sparsity
instances**, density `1−S` log-spaced from 1 (fully dense) down to 1/20, importance ≡ 1.
Every feature is identical — same importance, same sparsity — so any structure in the
solution is structure the *loss landscape* chose, not structure we baked in.

Two readouts per instance: `D* = m/‖W‖²_F` (average dims per learned feature, one point
per instance) and the full per-feature scatter of `Dᵢ` (200 points per instance). If
superposition were amorphous, the scatter would smear. The paper's claim: it clings to
a handful of exact fractions.

In [ ]:
# Act 3a: smoke test — time 100 steps before committing to 10k
import time

sweep_S = 1.0 - 20.0 ** -torch.linspace(0, 1, 20)   # density 1 → 1/20, log-spaced

torch.manual_seed(SEED)
_probe = BatchedToyModel(20, 200, 20)
t0 = time.time()
train_batched(_probe, sweep_S, steps=100, seed=SEED)
per_step = (time.time() - t0) / 100
print(f"{per_step*1000:.0f} ms/step → est. full run {per_step * 10_000 / 60:.1f} min")

In [ ]:
# Act 3b: the sweep — cached; delete cache/uniform_sweep.pt to retrain
CACHE.mkdir(exist_ok=True)
sweep_path = CACHE / "uniform_sweep.pt"
if sweep_path.exists():
    blob = torch.load(sweep_path)
    sweep_W, sweep_losses = blob["W"], blob["losses"]
    print("loaded cache:", sweep_path)
else:
    torch.manual_seed(SEED)
    sweep_model = BatchedToyModel(20, 200, 20)
    log = train_batched(sweep_model, sweep_S, steps=10_000, seed=SEED)
    sweep_W = sweep_model.W.detach().clone()
    sweep_losses = log["losses"]
    torch.save({"W": sweep_W, "losses": sweep_losses, "S": sweep_S, "seed": SEED}, sweep_path)
    print("trained and cached:", sweep_path)
print("final summed loss:", f"{sweep_losses[-1][1]:.4f}")

In [ ]:
# lib: plot_dimensionality_sweep
PLATEAUS = [
    (1.0, "1 — dedicated"),
    (3 / 4, "¾ — tetrahedron"),
    (2 / 3, "⅔ — triangle"),
    (1 / 2, "½ — digon"),
    (2 / 5, "⅖ — pentagon"),
    (3 / 8, "⅜ — square antiprism"),
]

def plot_dimensionality_sweep(sparsities, W, ax=None):
    """Per-feature Dᵢ scatter + D* line vs 1/(1−S), log-x, plateau guides. W [I, m, n]."""
    ax = ax or plt.gca()
    x = 1.0 / (1.0 - sparsities.cpu().numpy())
    for frac, name in PLATEAUS:
        ax.axhline(frac, lw=0.5, ls="--", color="gray")
        ax.annotate(name, (x[-1] * 1.08, frac), fontsize=7, va="center",
                    annotation_clip=False)
    dstars = []
    for i in range(W.shape[0]):
        D = feature_dimensionality(W[i]).numpy()
        jitter = x[i] * (1 + np.random.default_rng(i).uniform(-0.03, 0.03, len(D)))
        ax.scatter(jitter, D, s=3, alpha=0.35, color="tab:blue", linewidths=0)
        dstars.append(frobenius_dims_per_feature(W[i]))
    ax.plot(x, dstars, color="black", lw=1.5, marker="o", ms=3, label="D* = m/‖W‖²_F")
    ax.set_xscale("log")
    ax.set_xlabel("1/(1−S)  (sparser →)")
    ax.set_ylabel("feature dimensionality Dᵢ")
    ax.set_ylim(-0.05, 1.1)
    ax.legend(loc="upper right")

In [ ]:
# Act 3c: the payoff plot
fig, ax = plt.subplots(figsize=(9, 5.5))
plot_dimensionality_sweep(sweep_S, sweep_W, ax=ax)
ax.set_title("Dimensionality clings to fractions — superposition is quantized")
plt.tight_layout(); plt.show()

# one-line numeric claim: fraction of learned features within 0.02 of a plateau value
D_all = torch.cat([feature_dimensionality(sweep_W[i]) for i in range(sweep_W.shape[0])])
learned = D_all[D_all > 0.05]
plateau_vals = torch.tensor([f for f, _ in PLATEAUS])
dist = (learned[:, None] - plateau_vals).abs().min(dim=1).values  # nearest plateau only
near = int((dist < 0.02).sum())
print(f"{near}/{len(learned)} learned features ({near/len(learned):.0%}) sit within "
      f"0.02 of a named plateau (1, ¾, ⅔, ½, ⅖, ⅜)")

Whatever fraction printed is the result. The paper's version of this plot shows dense
bands exactly at ½ and ⅖ with sparser bands at ¾, ⅔, ⅜; stragglers *between* plateaus
are real too — the paper attributes them to imperfect convergence and non-uniform
polytopes, and Act 5 will show solutions migrating between levels mid-training. If a
named plateau (⅜ especially) is missing at this seed and scale, that is a finding to
report, not to hide.

## Act 4 — The plateaus are polytopes

A plateau at p/q should mean: *q features sharing p dimensions*. To check, split each
model's features into groups that interfere with each other but not with anyone else —
connected components of the interference graph (nodes: features with non-trivial norm;
edges: |cos(Wᵢ, Wⱼ)| above a small threshold). The paper calls these *tegum factors*:
the model tiles its 20 dimensions with small independent polytopes in orthogonal
subspaces.

Each component spans its own low-dimensional subspace, so we can PCA it down and simply
*look*. Predictions, one per plateau: ½ → digon (2 features, 1 dim), ⅔ → triangle
(3-in-2), ¾ → tetrahedron (4-in-3), ⅖ → pentagon (5-in-2), ⅜ → square antiprism
(8-in-3). And the Gram matrix gives the angles numerically — a regular tetrahedron's
off-diagonal cosines are exactly −⅓.

In [ ]:
# lib: interference_components
def interference_components(W, norm_cutoff=0.5, cos_threshold=0.15):
    """Connected components of the interference graph over represented features.

    Nodes: features with column norm > norm_cutoff. Edges: |cos(Wᵢ, Wⱼ)| > cos_threshold.
    Returns components as sorted index lists, largest first. W [m, n].
    """
    W = W.detach()
    norms = W.norm(dim=0)
    nodes = [i for i in range(W.shape[1]) if norms[i] > norm_cutoff]
    if not nodes:
        return []
    Wu = W[:, nodes] / norms[nodes]
    adj = (Wu.T @ Wu).abs() > cos_threshold
    comps, seen = [], set()
    for s in range(len(nodes)):
        if s in seen:
            continue
        stack, comp = [s], []
        while stack:
            u = stack.pop()
            if u in seen:
                continue
            seen.add(u)
            comp.append(nodes[u])
            stack.extend(v for v in range(len(nodes)) if adj[u, v] and v not in seen)
        comps.append(sorted(comp))
    return sorted(comps, key=len, reverse=True)

In [ ]:
# lib: project_component
def project_component(W, component, d=2):
    """PCA a component's feature columns into their own top-d subspace. Returns [d, k]."""
    Wc = W.detach()[:, component]                    # [m, k]
    U, _, _ = torch.linalg.svd(Wc, full_matrices=False)
    return U[:, :d].T @ Wc

In [ ]:
# lib: plot_component_2d
def plot_component_2d(W, component, ax=None, title=None):
    """Star plot of a component's features in their own 2-D PCA plane."""
    coords = project_component(W, component, d=2).cpu().numpy()
    ax = ax or plt.gca()
    for i in range(coords.shape[1]):
        ax.plot([0, coords[0, i]], [0, coords[1, i]], marker="o")
    lim = float(np.abs(coords).max()) * 1.2 + 1e-9
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.set_aspect("equal")
    ax.set_xticks([]); ax.set_yticks([])
    if title:
        ax.set_title(title, fontsize=9)

In [ ]:
# Act 4a: census of component sizes across the sweep, then a gallery
from collections import Counter

census = {}
for i in range(sweep_W.shape[0]):
    sizes = Counter(len(c) for c in interference_components(sweep_W[i]))
    census[f"1/(1-S)={1/(1-sweep_S[i]):.1f}"] = dict(sorted(sizes.items()))
for k, v in census.items():
    print(f"{k:16s} component sizes → {v}")

In [ ]:
# Act 4b: gallery — for each named plateau, find a component whose mean Dᵢ is nearest
targets = [(1 / 2, "digon"), (2 / 3, "triangle"), (3 / 4, "tetrahedron"),
           (2 / 5, "pentagon"), (3 / 8, "square antiprism")]
fig, axes = plt.subplots(1, len(targets), figsize=(3 * len(targets), 3.2))
found = {}
for ax, (frac, name) in zip(axes, targets):
    best = None  # (|meanD − frac|, instance, component)
    for i in range(sweep_W.shape[0]):
        D = feature_dimensionality(sweep_W[i])
        for comp in interference_components(sweep_W[i]):
            gap = abs(D[comp].mean().item() - frac)
            if best is None or gap < best[0]:
                best = (gap, i, comp)
    gap, i, comp = best
    found[name] = (i, comp, gap)
    mean_D = feature_dimensionality(sweep_W[i])[comp].mean().item()
    plot_component_2d(sweep_W[i], comp, ax=ax,
                      title=f"{name}\nk={len(comp)}, mean D = {mean_D:.3f}")
fig.suptitle("Components nearest each plateau, PCA-projected to their own plane")
plt.tight_layout(); plt.show()

In [ ]:
# Act 4c: the angles, numerically — Gram matrix of the tetrahedron-candidate component
i, comp, gap = found["tetrahedron"][0], found["tetrahedron"][1], found["tetrahedron"][2]
Wc = sweep_W[i][:, comp]
Wu = Wc / Wc.norm(dim=0)
G = (Wu.T @ Wu)
print(f"tetrahedron candidate: instance {i}, features {comp}, mean D gap {gap:.3f}")
print("pairwise cosines (regular tetrahedron → −0.333):")
print(np.array2string(G.numpy(), precision=3, suppress_small=True))

Read the gallery honestly. A 5-vertex component whose PCA plane shows five evenly-spaced
rays *is* a pentagon; an 8-vertex component at D≈⅜ lives in 3 dims, so its 2-D shadow
looks like two nested squares — that shadow plus the size-8/D=⅜ signature is the
antiprism evidence, and if no such component formed at this seed, the gallery shows
whatever did. Note what the paper predicts about tegum splits: instead of e.g. a
triangular bipyramid at ⅗, models prefer co-occurring triangles (⅔) and digons (½) —
which is why the census skews to small components.